# Day 7 - Code breaks, block 2: generative models (VAE and GAN)

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the value marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough** (the GAN takes 1-2 minutes). You cannot break anything - if something looks strange:
*Runtime → Restart session*, then run the first two cells again.

Data: **FashionMNIST** again - 28 × 28 grey pictures of clothes. Now we want **new** clothes that are not in the dataset.

## Setup - run these two cells first

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.utils import make_grid

train_set = datasets.FashionMNIST("data", train=True, download=True)
X = train_set.data.float().view(-1, 784) / 255          # 60 000 pictures, 784 pixels each, values 0 ... 1
print("training pictures:", tuple(X.shape))

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `make_autoencoder(bottleneck)` - the encoder and decoder from block 1 (784 → 256 → 64 → latent space → 64 → 256 → 784)
- `train(models, loss_fn, data, epochs)` - the training loop from day 3 (Adam, batches of 256), prints the loss
- `stay_close(centre, log_spread)` - the second part of the VAE loss: small when every cloud sits near 0 with spread 1
  (the formula is on the deep-dive slide "the VAE loss")
- `show(pictures, title)` - draws pictures in a grid
- `generator`, `discriminator` - the two networks of the GAN, with their optimizers

In [ ]:
def make_autoencoder(bottleneck_in, bottleneck_out=None):
    torch.manual_seed(0)
    encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, bottleneck_in))
    decoder = nn.Sequential(nn.Linear(bottleneck_out or bottleneck_in, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(),
                            nn.Linear(256, 784), nn.Sigmoid())
    return encoder, decoder


def train(models, loss_fn, data, epochs, batch_size=256):
    params = [p for m in models for p in m.parameters()]
    opt = torch.optim.Adam(params, lr=1e-3)
    for epoch in range(epochs):
        total = 0.0
        for idx in torch.randperm(len(data)).split(batch_size):
            loss = loss_fn(data[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f"epoch {epoch + 1}: loss {total / len(data):.2f}")


def stay_close(centre, log_spread):
    return -0.5 * (1 + log_spread - centre ** 2 - log_spread.exp()).sum(dim=1).mean()


def show(imgs, title="", nrow=10):
    grid = make_grid(imgs.detach().view(-1, 1, 28, 28), nrow=nrow, padding=2, pad_value=1)[0]
    plt.figure(figsize=(nrow * 0.8, 0.8 * ((len(imgs) - 1) // nrow + 1) + 0.4))
    plt.imshow(grid, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.title(title)
    plt.show()


torch.manual_seed(0)
generator = nn.Sequential(nn.Linear(64, 256), nn.LeakyReLU(0.2), nn.Linear(256, 512), nn.LeakyReLU(0.2),
                          nn.Linear(512, 784), nn.Tanh())                     # 64 random numbers → a picture
discriminator = nn.Sequential(nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Linear(512, 256), nn.LeakyReLU(0.2),
                              nn.Linear(256, 1))                              # a picture → one score: real or fake?
opt_G = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
X_gan = X * 2 - 1                                                             # pixels from -1 to 1 (Tanh output)
fixed_noise = torch.randn(20, 64)                                             # the same 20 random inputs after every epoch
print("helpers ready")

## 💻 Code break 4 - The VAE: a map without gaps
The encoder gives a **centre** and a **spread** - a small cloud instead of a point. The decoder gets a random point
from that cloud. The loss has **two parts**: rebuild the picture, and keep all clouds close to 0.

In [ ]:
KL_WEIGHT = 1.0      # 👈 try 0 and 10

encoder, decoder = make_autoencoder(4, 2)      # encoder: 2 numbers centre + 2 numbers spread; decoder: 2 numbers

def vae_loss(x):
    centre, log_spread = encoder(x).chunk(2, dim=1)
    z = centre + torch.randn_like(centre) * (0.5 * log_spread).exp()   # a random point from the cloud
    rebuild = ((decoder(z) - x) ** 2).sum(dim=1).mean()                # 1) rebuild the picture
    return rebuild + KL_WEIGHT * stay_close(centre, log_spread)        # 2) clouds stay close to 0

train([encoder, decoder], vae_loss, X, epochs=10)

with torch.no_grad():
    new = decoder(torch.randn(20, 2))                                   # 20 random points → 20 NEW pictures
    v = torch.linspace(-2, 2, 10)
    walk = decoder(torch.stack(torch.meshgrid(v, -v, indexing="xy"), -1).view(-1, 2))
show(new, "20 new pictures from random points in the latent space")
show(walk, "a walk over the latent space: dimension 1 from left to right, dimension 2 from top to bottom")

🔧 **Change it:** `KL_WEIGHT = 0` (= a normal autoencoder), then `KL_WEIGHT = 10`.

<details><summary>👀 What you should see</summary>

- `KL_WEIGHT = 1`: the random points give **new, plausible** clothes - not copies of training pictures. The walk shows
  one kind of clothing **morphing smoothly** into the next (trousers → T-shirts → pullovers, sneakers → boots → bags)
- all pictures are a bit **blurry** - typical for a VAE (when it is unsure, it paints the average)
- `KL_WEIGHT = 0` (= a normal autoencoder): the points can go anywhere, the map is much larger than -2 ... 2 and has
  gaps → random points give mostly **one kind** (shoes), dark or strange pictures - the walk shows almost only shoes
- `KL_WEIGHT = 10`: the clouds are pushed together → still clothes, but **even blurrier**: details such as heels,
  stripes or bag handles are gone. The loss part "stay close" wins over "rebuild"
</details>

## 💻 Code break 5 - The GAN: forger against police
The **generator** turns 64 random numbers into a picture. The **discriminator** gets real and fake pictures and says
"real" (1) or "fake" (0). They train in turns. Watch the same 20 random inputs after every epoch.

In [ ]:
EPOCHS = 5           # 👈 run the cell again: it trains 5 MORE epochs

for epoch in range(EPOCHS):
    for idx in torch.randperm(len(X_gan)).split(128):
        real = X_gan[idx]
        fake = generator(torch.randn(len(idx), 64))
        ones, zeros = torch.ones(len(idx), 1), torch.zeros(len(idx), 1)
        # 1) police: real → 1, fake → 0   (detach: this step must not change the forger)
        loss_D = bce(discriminator(real), ones) + bce(discriminator(fake.detach()), zeros)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # 2) forger: wants the police to say 1 for its fakes
        loss_G = bce(discriminator(fake), ones)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    with torch.no_grad():
        show((generator(fixed_noise) + 1) / 2, f"epoch {epoch + 1}: loss police {loss_D:.2f}, loss forger {loss_G:.2f}")

🔧 **Change it:** run the cell **again** (5 more epochs). Then change `fixed_noise` to `torch.randn(20, 64)` in the
`show(...)` line - new random inputs, new clothes.

<details><summary>👀 What you should see</summary>

- epoch 1: noise with a bright blob in the middle; epoch 3: rough shapes of tops; epoch 5: tops, pullovers and
  trousers can be recognised, shoes are noisy stripes - everything is still very **noisy**
- run the cell 3 more times (20 epochs, as on the slide): clothes of all kinds with hard edges and texture - not
  blurry like the VAE, but noisy; some pictures are broken, and a picture can change a lot from one epoch to the next
- the losses **do not go down steadily** - they go up and down, because each network makes the task harder for the
  other one. A GAN loss does **not** tell you how good the pictures are
- real GANs (StyleGAN) use convolutions, much more data and days of GPU time - the idea is the same
</details>